## Set Up NDIF

### Installing nnsight packages

In [ ]:
# Install nnsight hackathon version
!pip install git+https://github.com/ndif-team/nnsight.git@hackathon/peft
!pip install --upgrade torchao transformers

from IPython.display import clear_output
clear_output()

### Loading a Language Model

While we could define and create a model to pass in directly, LanguageModel includes special support for HuggingFace language models - it automatically loads the model AND the appropriate tokenizer from a HuggingFace ID.

Under the hood, LanguageModel uses AutoModelForCausalLM.from_pretrained() to load the model. Any keyword arguments you pass are forwarded directly to HuggingFace, so you can use all the same options:

```python
# Example with common HuggingFace kwargs:
model = LanguageModel(
    "meta-llama/Llama-3.1-8B-Instruct",
    device_map="auto",           # Distribute across GPUs
    torch_dtype=torch.float16,   # Set precision
    trust_remote_code=True,      # For custom model code
)
```

A note on model initialization:

- device_map="auto" tells HuggingFace Accelerate to automatically distribute model layers across all available GPUs (and CPU if the model doesn't fit). This is the recommended setting for large models.
- By default, nnsight uses lazy loading - the model isn't loaded into memory until the first trace. Pass dispatch=True to load immediately.

In [ ]:
from nnsight import LanguageModel

llm = LanguageModel("openai-community/gpt2", device_map="auto", dispatch=True)

print(llm)

### Loading LORAs from HuggingFace by PEFT ID

PEFT (parameter-efficent fine-tuning) is a library by huggingface for fine-tuning transformer models including training LORA layers. 

During the Red-Teaming part of this competion, teams trained LORAs and uploaded them to the NDIF huggingface. We can now add a `peft_id` to our `LanguageModel` definition to download LORA and apply the adapter for local or remote execution. 

In [ ]:
model_id = "meta-llama/Llama-3.1-8B-Instruct"
peft_id = "NDIF/hackathon-imposter-syndrome-eve-llama8B"

model = LanguageModel(model_id, peft=peft_id)

with model.trace("Hello world", remote=True) as tracer:

    output = model.output.save()

Some competition models are `VisionLanguage` models which accept LORAs the same way.

In [ ]:
from nnsight import VisionLanguageModel

model_id = "Qwen/Qwen3.5-27B"
peft_id = "NDIF/hackathon-imposter-eve-qwen9B"

model = VisionLanguageModel(model_id, peft_id=peft_id)

## Remote Execution of NDIF Models

nnsight can run interventions on large models hosted by the National Deep Inference Fabric (NDIF). Everything works the same - just add `remote=True`.

### Setup
Get your API key at https://login.ndif.us

In [ ]:
from nnsight import CONFIG

CONFIG.set_default_api_key("YOUR_API_KEY")

#### > Point to Hackathon Server

Competition models and LORAs are hosted on a different cloud instance of NDIF for increased hackathon capactity. You should have sent your API key to the NDIF team via the Blue Team Application for access to the hackathon server already.

Point to the Hackathon server by setting `CONFIG.API.HOST`

In [ ]:
CONFIG.API.HOST = "http://ndif-hackathon.duckdns.org:8001"

#### Hackathon Competition Models:
trusted:
- `Qwen/Qwen3.5-9B`

suspect:
- `google/gemma-3-27b-it`
- `Qwen/Qwen3.5-27B`
- `nvidia/Nemotron-3-Super-120B`

### Remote Tracing

In [ ]:
import os
os.environ['HF_TOKEN'] = "YOUR_HUGGING_FACE_TOKEN"

llama_model = LanguageModel("meta-llama/Meta-Llama-3.3-70B-Instruct", device_map="auto", dtype="bfloat16")

# Just add remote=True - everything else is the same!
with llama_model.trace("The Eiffel Tower is in the city of", remote=True):
    hidden_states = llama_model.model.layers[-1].output.save()
    output = llama_model.output.save()

print("Hidden states shape:", hidden_states[0].shape)

### Multi-Token Generation

So far we've done single forward passes. But language models generate text by running multiple forward passes - one per token. This means the same modules are called multiple times!

Use `.generate()` instead of `.trace()` for multi-token generation:

In [ ]:
with llama_model.generate("The Eiffel Tower is in", max_new_tokens=3) as tracer:
    output = llama_model.generator.output.save()

print(llama_model.tokenizer.decode(output[0]))

### Applying a Chat Template

In order to apply a chat template, there is a specific format that each conversation piece should take. They should be formatted as a list of dictionaries with `role` and `content` key value pairs.

The `role` key is used to specify the speaker and the `content` key is used to describe how the model should respond when prompted by the given speaker. In order to apply the template we need to load the tokenizer from the LLama 70B Instruct model so that the model can convert tokens into text. The `user` role is the human asking the question and the `assistant` role provides context to the model about what has already been "said".

Now we can pass a messager to our model and see how it responds!

In [ ]:
from transformers import AutoTokenizer
# load in tokenizer
tokenizer = AutoTokenizer.from_pretrained("meta-llama/Llama-3.3-70B-Instruct")

# define chat conversation
chat = [
    {"role": "system", "content": "You are a friendly chatbot who always responds like a teacher"},
    {"role": "user", "content": "How many helicopters can a human eat in one sitting?"},
 ]

# convert the conversation into a format the model will understand
prompt = tokenizer.apply_chat_template(chat, tokenize=True, add_generation_prompt=True)

print(tokenizer.decode(prompt))

In [ ]:
with llama_model.generate(prompt, max_new_tokens=128, remote=True) as gen:
    # save final generated tokens
    saved = model.generator.output.save()

# print each decoded output on a new line
for seq in saved:
    print(llama_model.tokenizer.decode(seq, skip_special_tokens=True))

### Skills Repository

**Claude Code**

```bash
# Open Claude Code terminal
claude

# Add the marketplace (one time)
/plugin marketplace add https://github.com/ndif-team/skills.git

# Install all skills
/plugin install nnsight@skills
```

**OpenAI Codex**

```bash
# Open OpenAI Codex terminal
codex

# Install skills
skill-installer install https://github.com/ndif-team/skills.git
```